In [ ]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [ ]:
dbutils.widgets.text("catalog_name","projectecommerce","Catalog name")
dbutils.widgets.text("schema_account_name","stdecommercedevcink001", "Storage Account Name")
dbutils.widgets.text("Container_name","raw-ecomm-data-ci", "Container Name")

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("schema_account_name")
container_name = dbutils.widgets.get("Container_name")

In [ ]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"

silver_table_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/fact_order_items/"

In [ ]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
df = spark.readStream \
    .format("delta") \
    .table(f"{catalog_name}.bronze.brz_order_items")

# Transformations
df = df.dropDuplicates(["order_id", "item_seq"])

df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2)
    .otherwise(F.col("quantity"))
    .cast("int")
)

df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
     .when(F.col("channel") == "app", "Mobile")
     .otherwise(F.col("channel"))
)

df = df.withColumn("processed_time", F.current_timestamp())

# Streaming write
df.writeStream \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", silver_checkpoint_path) \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()